# FIBA Tile Montage (MATLAB-modeled) — walkthrough

This notebook documents (and partially re-implements in Python) the processing pipeline used by the Fiji/ImageJ plugin **FIBA Tile Montage (MATLAB)**.

It is meant for troubleshooting and clarity: each step is a separate cell so you can see intermediate outputs (images/plots), and the math behind key steps is written out.

The Java plugin is the source of truth; the Python pieces here are a transparency tool to visualize what each stage is doing.

## Pipeline overview

For each tile (square crop), the Java pipeline does:

1. Contrast stretch (MATLAB `imadjust`-like).
2. Apply a separable 2D Tukey window (edge blending).
3. Compute 2D FFT, take magnitude, then `fftshift`.
4. Build an orientation signal $\text{SOL}(\theta)$ by sampling in polar coordinates and summing along radius.
5. Find a statistically significant peak band and estimate fiber angle.
6. Build a band-limited frequency mask (radial Tukey × angular Tukey), inverse FFT reconstruct, threshold.

Outputs (typical): montage, per-tile panels (`*_crop.jpg`, `*_fft.jpg`, `*_fft.tif`, `*_polar.jpg`, `*_rec.jpg`, `*_mask.jpg`), SOL plots, CSV.

## Quarto (markdown-first) version

A Quarto version of this walkthrough lives at `notebooks/fiba_tile_montage_pipeline.qmd`.

- It contains the same equations and the same ‘load outputs from Downloads’ diagnostic code blocks.
- Rendering defaults to *not executing* code so it builds even without Python/Jupyter installed.
- Rendering settings are in the repo root `_quarto.yml`.

## Key equations and concepts

### Tukey window (MATLAB `tukeywin`)
Let $n$ be the length and $\alpha \in [0,1]$. Define $t = \frac{i}{n-1}$ for $i=0,\dots,n-1$.

$$
w(t)=\begin{cases}
\tfrac12\left(1+\cos\left(\pi\left(\tfrac{2t}{\alpha}-1\right)\right)\right), & 0\le t < \tfrac{\alpha}{2}\\
1, & \tfrac{\alpha}{2} \le t \le 1-\tfrac{\alpha}{2}\\
\tfrac12\left(1+\cos\left(\pi\left(\tfrac{2t}{\alpha}-\tfrac{2}{\alpha}+1\right)\right)\right), & 1-\tfrac{\alpha}{2} < t \le 1
\end{cases}
$$

The plugin applies this separably as a 2D window $S_{r,c} = w_r w_c$.

### Discrete Fourier Transform (DFT)
For an $n\times n$ image $S[x,y]$ (after normalization + windowing), the 2D DFT is
$$
K[u,v] = \sum_{x=0}^{n-1} \sum_{y=0}^{n-1} S[x,y] \exp\left(-j2\pi\left(\frac{ux}{n} + \frac{vy}{n}\right)\right)
$$
for frequency indices $u,v \in \{0,\dots,n-1\}$.

#### How the FFT relates
The **FFT** is not a different transform; it is an algorithm that computes the **exact same DFT values** as the sums above, but faster.
In 1D it reduces complexity from $O(N^2)$ to $O(N\log N)$; for a 2D $n\times n$ image it is typically $O(n^2\log n)$ via separable 1D FFTs along rows and columns.

In signal-processing terms: the DFT/FFT is a *discrete* transform. It approximates a continuous Fourier transform only in the modeling sense that we treat $S[x,y]$ as samples of an underlying continuous image.

### FFT magnitude and display scaling
Given complex FFT $K(u,v)$, the magnitude is $|K|=\sqrt{\Re(K)^2+\Im(K)^2}$. The plugin uses `fftshift` so DC is centered.

The **power spectrum** (what we plot on a log scale) is
$$
P(u,v) = |K(u,v)|^2.
$$
For visualization, the plugin uses a robust log-power display: $D = \log(1 + P)$, then a percentile-based contrast stretch so a single spike does not dominate.

### Orientation signal (SOL)
With center at $(w,w)$ where $w=n/2$ and $\theta$ measured from the **vertical axis** (row direction):
$$
B(\theta)=\sum_{r=r_{min}}^{r_{max}} |K|\big(w + r\cos\theta,\; w + r\sin\theta\big)
$$
using bilinear sampling. $B$ is then mapped into 180 bins and normalized to form $\text{SOL}(\theta)$ with $\sum \text{SOL}=1$.

In [ ]:
# Imports
from __future__ import annotations

from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (6, 6)
plt.rcParams['image.cmap'] = 'gray'
print('OK')

In [ ]:
# Configure where Fiji/Java outputs are located
downloads = Path.home() / 'Downloads'
out_dir = downloads

# Base name used by FIBA_Tile_Montage = stripExtension(imp.getTitle())
# For an image opened as .../C15D5P001_tiles/4.jpg, baseName is typically '4'.
base = '4'

print('Output dir:', out_dir)
print('Exists:', out_dir.exists())

In [ ]:
# List the newest output files for this base name
files = sorted(out_dir.glob(f'{base}_tile*'), key=lambda p: p.stat().st_mtime, reverse=True)
for p in files[:30]:
    print(p.name)
print('Total matching files:', len(files))

In [ ]:
# Show montage and the orientation-vs-tile profile
montage_path = out_dir / f'{base}_tile_montage.jpg'
profile_path = out_dir / f'{base}_tile_profile.jpg'

for title, path in [('montage', montage_path), ('profile', profile_path)]:
    if path.exists():
        img = Image.open(path)
        plt.figure(figsize=(10, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(f'{title}: {path.name}')
        plt.show()
    else:
        print('Missing:', path)

In [ ]:
# Inspect one tile's outputs
tile_id = 4
paths = {
    'crop': out_dir / f'{base}_tile{tile_id}_crop.jpg',
    'fft_jpg': out_dir / f'{base}_tile{tile_id}_fft.jpg',
    'fft_tif': out_dir / f'{base}_tile{tile_id}_fft.tif',
    'polar': out_dir / f'{base}_tile{tile_id}_polar.jpg',
    'rec': out_dir / f'{base}_tile{tile_id}_rec.jpg',
    'mask': out_dir / f'{base}_tile{tile_id}_mask.jpg',
}

for k, p in paths.items():
    print(k, '->', p.exists(), p.name)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()
keys = ['crop', 'fft_jpg', 'polar', 'mask', 'rec']
for ax, k in zip(axes, keys):
    p = paths[k]
    if not p.exists():
        ax.set_title(f'missing: {k}')
        ax.axis('off')
        continue
    ax.imshow(Image.open(p))
    ax.set_title(k)
    ax.axis('off')
axes[-1].axis('off')
plt.tight_layout()
plt.show()

## More math: masks, shifting, and why axis artifacts happen

### `fftshift` / `ifftshift`
The FFT output is indexed with DC at the origin. `fftshift` permutes quadrants so DC is in the center (useful for display and for polar sampling around the center).
For even-sized images ($n$ even), `fftshift` and `ifftshift` are identical permutations; for odd sizes they differ by one sample.

### Reconstruction mask (conceptual)
The Java pipeline builds a frequency-domain mask as a product of two Tukey-like tapers:

- radial taper $A(r)$ that passes $r n [r_{min}, r_{max}]$ with smooth edges controlled by $β$
- angular taper $B(θ)$ that passes the detected peak band $[θ_1,θ_2]$ with smooth edges controlled by $γ$

Then (conceptually) $M(r,θ) = A(r)B(θ)$, symmetrized so the mask respects Fourier conjugate symmetry.

### A common gotcha (now fixed in Java)
If the FFT library expects a particular memory layout and the input is written with the wrong stride (e.g., inserting zeros between samples), the spectrum can show missing/chunked energy along axes. The Java code was updated so the real input is written in the layout that `realForwardFull` expects.

## Diagnosing FFT display artifacts (JPEG vs lossless TIFF)

The plugin can save `*_fft.tif` (32-bit float) when `saveFftTif=true`.
This helps determine whether visible `clipping/gaps` are introduced by JPEG compression or 8-bit quantization.

Below we plot the center-column profile for the JPEG, and optionally for the TIFF.

In [ ]:
def read_grayscale_u8(path: Path) -> np.ndarray:
    img = Image.open(path).convert('L')
    return np.asarray(img, dtype=np.float32) / 255.0

fft_jpg = paths['fft_jpg']
jpg = read_grayscale_u8(fft_jpg)
h, w = jpg.shape
cc = jpg[:, w // 2]
print('FFT JPG shape:', jpg.shape)
print('center-col zeros:', int(np.sum(cc == 0.0)), 'of', cc.size)

plt.figure(figsize=(10, 3))
plt.plot(cc)
plt.title('Center column profile (FFT JPG)')
plt.ylim(-0.02, 1.02)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
fft_tif = paths['fft_tif']
if not fft_tif.exists():
    print('No TIFF found:', fft_tif)
else:
    try:
        import tifffile as tiff
        tif = tiff.imread(str(fft_tif)).astype(np.float32)
        cc_t = tif[:, tif.shape[1] // 2]
        print('FFT TIF shape:', tif.shape)
        print('center-col exact zeros (float):', int(np.sum(cc_t == 0.0)), 'of', cc_t.size)

        lo, hi = np.quantile(tif, [0.01, 0.999])
        tif_vis = np.clip((tif - lo) / (hi - lo + 1e-12), 0, 1)
        plt.figure(figsize=(6, 6))
        plt.imshow(tif_vis)
        plt.axis('off')
        plt.title('FFT TIFF (visualized)')
        plt.show()

        plt.figure(figsize=(10, 3))
        plt.plot(cc_t)
        plt.title('Center column profile (FFT TIFF, raw float)')
        plt.grid(True, alpha=0.3)
        plt.show()
    except Exception as e:
        print('Could not read TIFF (install tifffile?):', e)

## Minimal Python re-implementation (for intuition)

This re-implements the conceptual steps on the saved crop tile:
- Tukey window (MATLAB-equivalent definition)
- FFT and fftshift magnitude
- log-power + percentile stretch for display
- SOL-like curve via polar sampling

This is not a replacement for the Java code; it is a didactic mirror so you can see intermediate arrays.

In [ ]:
def tukeywin(n: int, alpha: float) -> np.ndarray:
    if n <= 0: return np.array([], dtype=np.float64)
    if n == 1: return np.array([1.0], dtype=np.float64)
    a = float(np.clip(alpha, 0.0, 1.0))
    if a == 0.0: return np.ones(n, dtype=np.float64)
    N = n - 1
    t = np.arange(n, dtype=np.float64) / N
    w = np.ones(n, dtype=np.float64)
    a2 = a / 2.0
    left = t < a2
    mid = (t >= a2) & (t <= (1.0 - a2))
    right = t > (1.0 - a2)
    w[left] = 0.5 * (1.0 + np.cos(np.pi * ((2.0 * t[left] / a) - 1.0)))
    w[mid] = 1.0
    w[right] = 0.5 * (1.0 + np.cos(np.pi * ((2.0 * t[right] / a) - (2.0 / a) + 1.0)))
    return w

def spectrum_display01(imgF_shift_mag: np.ndarray, lo_q=0.01, hi_q=0.999) -> np.ndarray:
    v = np.log1p(np.square(imgF_shift_mag))
    v = v.copy()
    v[v.shape[0] // 2, v.shape[1] // 2] = np.min(v)
    lo, hi = np.quantile(v, [lo_q, hi_q])
    out = (v - lo) / (hi - lo + 1e-12)
    return np.clip(out, 0, 1)

crop_img = read_grayscale_u8(paths['crop'])
n = crop_img.shape[0]
alpha = 0.4
w1 = tukeywin(n, alpha)
W = np.outer(w1, w1)

j = (crop_img - crop_img.min()) / (crop_img.max() - crop_img.min() + 1e-12)
avg = float(j.mean())
imgS = W * (j - avg) + avg
imgS = imgS / (imgS.max() + 1e-12)

K = np.fft.fft2(imgS)
amp = np.abs(K)
imgF = np.fft.fftshift(amp)
imgF_disp = spectrum_display01(imgF)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(j); ax[0].set_title('crop (normalized)'); ax[0].axis('off')
ax[1].imshow(imgS); ax[1].set_title('after Tukey window (ImgS)'); ax[1].axis('off')
ax[2].imshow(imgF_disp); ax[2].set_title('FFT magnitude (display)'); ax[2].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
def bilinear(img: np.ndarray, r: float, c: float) -> float:
    h, w = img.shape
    r0 = int(np.floor(r)); r1 = int(np.ceil(r))
    c0 = int(np.floor(c)); c1 = int(np.ceil(c))
    r0 = int(np.clip(r0, 0, h - 1)); r1 = int(np.clip(r1, 0, h - 1))
    c0 = int(np.clip(c0, 0, w - 1)); c1 = int(np.clip(c1, 0, w - 1))
    dr = r - r0
    dc = c - c0
    v00 = img[r0, c0]; v10 = img[r1, c0]
    v01 = img[r0, c1]; v11 = img[r1, c1]
    return float((1 - dr) * (1 - dc) * v00 + dr * (1 - dc) * v10 + (1 - dr) * dc * v01 + dr * dc * v11)

def sol_curve(imgF_shift_mag: np.ndarray, rmin: int, rmax: int) -> np.ndarray:
    n = imgF_shift_mag.shape[0]
    w = n // 2
    out = np.zeros(180, dtype=np.float64)
    for theta_deg in range(180):
        th = np.deg2rad(theta_deg)
        s = 0.0
        for r in range(rmin, rmax + 1):
            rr = w + r * np.cos(th)
            cc = w + r * np.sin(th)
            s += bilinear(imgF_shift_mag, rr, cc)
        out[theta_deg] = s
    out = out / (out.sum() + 1e-12)
    return out

rmin, rmax = 4, (n // 2 - 2)
sol = sol_curve(imgF, rmin, rmax)
plt.figure(figsize=(12, 3))
plt.plot(sol)
plt.title('SOL-like curve (Python)')
plt.xlabel('angle (deg)')
plt.ylabel('normalized sum')
plt.grid(True, alpha=0.3)
plt.show()
print('peak angle (argmax):', int(np.argmax(sol)))